# ArcFace 评估 (Kaggle 版)

在 Kaggle 上评估 LFW / CFP-FP / CPLFW / AgeDB 等 1:1 验证集。

**前置条件**: 需要一个训练好的 `last.pt`(含 `backbone` state_dict)。三种方式之一:
1. **同一会话**: 先运行训练 notebook,`/kaggle/working/output/last.pt` 已生成, 直接跑本 notebook。
2. **上传模型**: 把本地 `last.pt` 拖到右侧 Data → `Add file`, 它会出现在 `/kaggle/working/`。
3. **存成数据集**: 训练完用「Create Dataset」把 `last.pt` 存成新数据集, 下次 `Add Input` 引用。

**验证数据已内置**: `/kaggle/input/datasets/debarghamitraroy/casia-webface/eval`

**注意**: 需开启 **GPU 加速器** (T4 即可)。

In [ ]:
%%writefile iresnet.py
"""
============================================================
IResNet (Improved ResNet) 人脸识别主干
insightface 风格: BN→Conv→BN→PReLU→Conv→BN 基本块, 专为 112×112 设计
(保持分辨率: stem 用 3x3 stride1, 不用 7x7 stride2 + maxpool)

输出 512 维 embedding (经 BatchNorm1d 归一化)
============================================================
"""
import torch
from torch import nn

__all__ = ['iresnet18', 'iresnet34', 'iresnet50', 'iresnet100']


def conv3x3(in_planes, out_planes, stride=1, groups=1, dilation=1):
    return nn.Conv2d(in_planes, out_planes, kernel_size=3, stride=stride,
                     padding=dilation, groups=groups, bias=False, dilation=dilation)


def conv1x1(in_planes, out_planes, stride=1):
    return nn.Conv2d(in_planes, out_planes, kernel_size=1, stride=stride, bias=False)


class IBasicBlock(nn.Module):
    expansion = 1

    def __init__(self, inplanes, planes, stride=1, downsample=None,
                 groups=1, base_width=64, dilation=1):
        super(IBasicBlock, self).__init__()
        if groups != 1 or base_width != 64:
            raise ValueError('IBasicBlock only supports groups=1 and base_width=64')
        if dilation > 1:
            raise NotImplementedError("Dilation > 1 not supported in IBasicBlock")
        self.bn1 = nn.BatchNorm2d(inplanes, eps=1e-05)
        self.conv1 = conv3x3(inplanes, planes)
        self.bn2 = nn.BatchNorm2d(planes, eps=1e-05)
        self.prelu = nn.PReLU(planes)
        self.conv2 = conv3x3(planes, planes, stride)
        self.bn3 = nn.BatchNorm2d(planes, eps=1e-05)
        self.downsample = downsample
        self.stride = stride

    def forward(self, x):
        identity = x
        out = self.bn1(x)
        out = self.conv1(out)
        out = self.bn2(out)
        out = self.prelu(out)
        out = self.conv2(out)
        out = self.bn3(out)
        if self.downsample is not None:
            identity = self.downsample(x)
        out += identity
        return out


class IResNet(nn.Module):
    fc_scale = 7 * 7  # 112×112 输入 → 最后一层特征图 7×7

    def __init__(self, block, layers, dropout=0, num_features=512,
                 zero_init_residual=False, groups=1, width_per_group=64,
                 replace_stride_with_dilation=None):
        super(IResNet, self).__init__()
        self.inplanes = 64
        self.dilation = 1
        if replace_stride_with_dilation is None:
            replace_stride_with_dilation = [False, False, False]
        self.groups = groups
        self.base_width = width_per_group

        self.conv1 = nn.Conv2d(3, self.inplanes, kernel_size=3, stride=1, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(self.inplanes, eps=1e-05)
        self.prelu = nn.PReLU(self.inplanes)

        self.layer1 = self._make_layer(block, 64, layers[0], stride=2)
        self.layer2 = self._make_layer(block, 128, layers[1], stride=2,
                                       dilate=replace_stride_with_dilation[0])
        self.layer3 = self._make_layer(block, 256, layers[2], stride=2,
                                       dilate=replace_stride_with_dilation[1])
        self.layer4 = self._make_layer(block, 512, layers[3], stride=2,
                                       dilate=replace_stride_with_dilation[2])

        self.bn2 = nn.BatchNorm2d(512 * block.expansion, eps=1e-05)
        self.dropout = nn.Dropout(p=dropout, inplace=True)
        self.fc = nn.Linear(512 * block.expansion * self.fc_scale, num_features)
        self.features = nn.BatchNorm1d(num_features, eps=1e-05)
        nn.init.constant_(self.features.weight, 1.0)
        self.features.weight.requires_grad = False

        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')
            elif isinstance(m, (nn.BatchNorm2d, nn.GroupNorm)):
                nn.init.constant_(m.weight, 1)
                nn.init.constant_(m.bias, 0)
        if zero_init_residual:
            for m in self.modules():
                if isinstance(m, IBasicBlock):
                    nn.init.constant_(m.bn3.weight, 0)

    def _make_layer(self, block, planes, blocks, stride=1, dilate=False):
        downsample = None
        previous_dilation = self.dilation
        if dilate:
            self.dilation *= stride
            stride = 1
        if stride != 1 or self.inplanes != planes * block.expansion:
            downsample = nn.Sequential(
                conv1x1(self.inplanes, planes * block.expansion, stride),
                nn.BatchNorm2d(planes * block.expansion, eps=1e-05),
            )
        layers = [block(self.inplanes, planes, stride, downsample,
                        self.groups, self.base_width, previous_dilation)]
        self.inplanes = planes * block.expansion
        for _ in range(1, blocks):
            layers.append(block(self.inplanes, planes, groups=self.groups,
                                base_width=self.base_width, dilation=self.dilation))
        return nn.Sequential(*layers)

    def forward(self, x):
        x = self.conv1(x)
        x = self.bn1(x)
        x = self.prelu(x)
        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)
        x = self.layer4(x)
        x = self.bn2(x)
        x = torch.flatten(x, 1)
        x = self.dropout(x)
        x = self.fc(x)
        x = self.features(x)
        return x


def _iresnet(arch, block, layers, **kwargs):
    return IResNet(block, layers, **kwargs)


def iresnet18(**kwargs):
    return _iresnet('iresnet18', IBasicBlock, [2, 2, 2, 2], **kwargs)


def iresnet34(**kwargs):
    return _iresnet('iresnet34', IBasicBlock, [3, 4, 6, 3], **kwargs)


def iresnet50(**kwargs):
    return _iresnet('iresnet50', IBasicBlock, [3, 4, 14, 3], **kwargs)


def iresnet100(**kwargs):
    return _iresnet('iresnet100', IBasicBlock, [3, 13, 30, 3], **kwargs)


In [ ]:
%%writefile kaggle_eval.py
"""
============================================================
ArcFace 人脸识别 1:1 验证集评估 (Kaggle 版, 可移植)
读取 insightface 官方 eval/*.bin (pickle: [jpeg_bytes列表, issame布尔列表])
  - lfw.bin      (LFW, 6000 pairs)   ← 验收标准 ≥ 99.5%
  - cfp_fp.bin   (CFP-Frontal-Profile)
  - cplfw.bin    (CPLFW)
  - agedb_30.bin (AgeDB-30)
  - 其余: calfw / cfp_ff / sllfw / talfw (参考)

iresnet.py 与本脚本同目录。

用法:
  python kaggle_eval.py --model output/last.pt --flip
============================================================
"""
import os
import sys
import pickle
import time
from io import BytesIO

SCRIPT_DIR = os.path.dirname(os.path.abspath(__file__))
sys.path.insert(0, SCRIPT_DIR)

# Kaggle 公开数据集默认 eval 目录
DEFAULT_EVAL_DIR = '/kaggle/input/datasets/debarghamitraroy/casia-webface/eval'


def arg(name, default):
    try:
        i = sys.argv.index(name)
        return sys.argv[i + 1]
    except (ValueError, IndexError):
        return default


def main():
    MODEL = arg('--model', os.path.join(SCRIPT_DIR, 'output/last.pt'))
    EVAL_DIR = arg('--data', DEFAULT_EVAL_DIR)
    BATCH = int(arg('--batch', 128))
    FLIP = '--flip' in sys.argv
    BINS = arg('--bins', '').split(',') if arg('--bins', '') else \
        ['lfw.bin', 'cfp_fp.bin', 'cplfw.bin', 'agedb_30.bin',
         'calfw.bin', 'cfp_ff.bin', 'sllfw.bin', 'talfw.bin']

    import numpy as np
    import cv2
    import torch
    from torch.utils.data import DataLoader, TensorDataset
    from torchvision import transforms

    from iresnet import iresnet50

    if not torch.cuda.is_available():
        print('[ERROR] CUDA not available'); sys.exit(1)
    device = torch.device('cuda')
    print('=' * 64)
    print(f'  Face Verification Eval  model={MODEL}')
    print(f'  flip={FLIP}  device={torch.cuda.get_device_name(0)}')
    print('=' * 64)

    # ---------------- 模型 ----------------
    ckpt = torch.load(MODEL, map_location='cpu', weights_only=False)
    num_classes = ckpt.get('num_classes', '?')
    epoch = ckpt.get('epoch', '?')
    backbone = iresnet50(num_features=512).to(device)
    backbone.load_state_dict(ckpt['backbone'])
    backbone.eval()
    print(f'  [model] epoch={epoch}  num_classes={num_classes}  (仅用 backbone)')

    # ---------------- 预处理 (与训练一致, 无随机) ----------------
    MEAN = [0.5, 0.5, 0.5]
    STD = [0.5, 0.5, 0.5]

    def decode_preprocess(jpeg_bytes):
        """JPEG bytes → [3,112,112] float tensor (RGB, 归一化)"""
        arr = np.frombuffer(jpeg_bytes, np.uint8)
        img = cv2.imdecode(arr, cv2.IMREAD_COLOR)          # BGR
        if img is None:
            img = np.zeros((112, 112, 3), np.uint8)
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        img = cv2.resize(img, (112, 112))
        t = torch.from_numpy(img).permute(2, 0, 1).float().div_(255.0)
        for c in range(3):
            t[c] = (t[c] - MEAN[c]) / STD[c]
        return t

    @torch.no_grad()
    def extract_features(images_tensor):
        """images_tensor: [N,3,112,112] → [N,512] L2 归一化"""
        feats = []
        n = images_tensor.size(0)
        for i in range(0, n, BATCH):
            x = images_tensor[i:i + BATCH].to(device)
            f = backbone(x)
            if FLIP:
                f = f + backbone(torch.flip(x, dims=[3]))
                f = f / 2.0
            f = torch.nn.functional.normalize(f, p=2, dim=1)
            feats.append(f.cpu())
        return torch.cat(feats, dim=0)

    def calc_accuracy(sims, issame):
        """余弦相似度 sims[配对], issame[bool] → 最优阈值下的 accuracy"""
        sims = sims.numpy().astype(np.float64)
        issame = np.asarray(issame, dtype=bool)
        best_acc, best_thresh = 0.0, 0.0
        for thresh in np.arange(-1.0, 1.0, 0.005):
            pred = sims > thresh
            acc = (pred == issame).mean()
            if acc > best_acc:
                best_acc, best_thresh = acc, thresh
        return best_acc, best_thresh

    # ---------------- 逐数据集评估 ----------------
    print(f'\n  {"数据集":>10s} | {"pairs":>6s} | {"acc":>7s} | {"阈值":>7s} | {"耗时":>7s}')
    print('  ' + '-' * 58)
    for bin_name in BINS:
        bin_path = os.path.join(EVAL_DIR, bin_name)
        if not os.path.exists(bin_path):
            print(f'  {bin_name:>10s} |  (跳过, 文件不存在)')
            continue
        t0 = time.time()
        with open(bin_path, 'rb') as f:
            bins, issame = pickle.load(f, encoding='bytes')
        n_pairs = len(issame)
        # 解码全部图片
        tensors = [decode_preprocess(b) for b in bins]
        imgs = torch.stack(tensors)                       # [2*n_pairs, 3,112,112]
        feats = extract_features(imgs)                    # [2*n_pairs, 512]
        feats = feats.view(n_pairs, 2, -1)
        sims = torch.nn.functional.cosine_similarity(feats[:, 0], feats[:, 1], dim=1)
        acc, thresh = calc_accuracy(sims, issame)
        el = time.time() - t0
        mark = '  ★ LFW' if bin_name == 'lfw.bin' else ''
        print(f'  {bin_name[:-4]:>10s} | {n_pairs:>6d} | {acc*100:>6.2f}% | {thresh:>7.3f} | {el:>6.1f}s{mark}')

    print('=' * 64)
    print('  Eval Complete.  (LFW 验收标准 ≥ 99.5%)')
    print('=' * 64)


if __name__ == '__main__':
    main()


In [ ]:
# 评估: 默认模型 /kaggle/working/output/last.pt, 加 --flip 镜像增强
!python kaggle_eval.py --model output/last.pt --flip

# 若模型在别处, 指定路径:
# !python kaggle_eval.py --model /kaggle/working/last.pt --flip